In [1]:
from torch import nn
import torch.nn.functional as F

class NonLinear(nn.Module):
    def __init__(self, input, output_size, hidden=None):
        super(NonLinear, self).__init__()
        if hidden is None:
            hidden = input
        self.layer1 = nn.Linear(input, hidden)
        self.layer2 = nn.Linear(hidden, output_size)

    def forward(self, x):
        x = F.gelu(self.layer1(x))
        x = self.layer2(x)
        return x

In [2]:
import torch
from torch import nn

@torch.jit.script
def gaussian(x, mean, std):
    pi = 3.14159
    a = (2*pi) ** 0.5
    return torch.exp(-0.5 * (((x - mean) / std) ** 2)) / (a * std)

class GaussianLayer(nn.Module):
    def __init__(self, K=128, edge_types=1024):
        super().__init__()
        self.K = K
        self.means = nn.Embedding(1, K)
        self.stds = nn.Embedding(1, K)
        self.mul = nn.Embedding(edge_types, 1)
        self.bias = nn.Embedding(edge_types, 1)
        nn.init.uniform_(self.means.weight, 0, 3)
        nn.init.uniform_(self.stds.weight, 0, 3)
        nn.init.constant_(self.bias.weight, 0)
        nn.init.constant_(self.mul.weight, 1)

    def forward(self, x, edge_types):
        mul = self.mul(edge_types)
        bias = self.bias(edge_types)
        x = mul * x.unsqueeze(-1) + bias
        x = x.expand(-1, -1, -1, self.K)
        mean = self.means.weight.float().view(-1)
        std = self.stds.weight.float().view(-1).abs() + 1e-5
        return gaussian(x.float(), mean, std).type_as(self.means.weight)

In [3]:
pos = torch.randn((2, 10, 3))
atoms = torch.randn((2, 10))
silica_mask = torch.randint(0, 2, atoms.shape).bool()
atoms[silica_mask] = 14
atoms[~silica_mask] = 8
atoms = atoms.long()

In [4]:
n_graph, n_node = atoms.size()
delta_pos = pos.unsqueeze(1) - pos.unsqueeze(2)
dist = delta_pos.norm(dim=-1)
delta_pos /= dist.unsqueeze(-1) + 1e-5

edge_type = atoms.view(
    n_graph, n_node, 1
) * 20 + atoms.view(n_graph, 1, n_node)

emb_n = 768
emb_e = 128
n_heads = 32
gbf = GaussianLayer(K=emb_e)
gbf_feature = gbf(dist, edge_type)
local_mask = 1 - 6 * (dist / 8) ** 5 + 15 * (dist / 8) ** 4 - 10 * (dist / 8) ** 3
local_mask.shape, gbf_feature.shape, delta_pos.shape

(torch.Size([2, 10, 10]),
 torch.Size([2, 10, 10, 128]),
 torch.Size([2, 10, 10, 3]))

In [5]:
batch, n, _, _ = delta_pos.shape

m_r_prod = local_mask.unsqueeze(-1) * delta_pos # shape: (batch, n, n, 3)
e_i_o = torch.sum(m_r_prod.unsqueeze(-1) * gbf_feature.unsqueeze(-2), dim=2)
e_i_o.shape

torch.Size([2, 10, 3, 128])

In [6]:
x = torch.randn((2, 10, emb_n))
bias = torch.randn((2, 32, 10, 10))

In [7]:
from matdeeplearn.models.model_dev.geomformer import GeomformerBlock

block = GeomformerBlock(768, 128, 32)
inv_out, equ_out = block(x, e_i_o, bias)

1: torch.Size([2, 32, 10, 10])
1: torch.Size([2, 32, 10, 10])
torch.Size([2, 32, 30, 30])
torch.Size([2, 32, 30, 30])


In [8]:
inv_out.shape, equ_out.shape

(torch.Size([2, 10, 768]), torch.Size([2, 10, 3, 128]))

torch.Size([2, 10])